# Module 04: Seaborn for Machine Learning
## Notebook 02: Relational and Categorical Visualizations

Machine learning datasets feature intricate multi-variable relationships. Seaborn excels at encoding up to 5 dimensions onto a single 2D plane using aesthetic channels (x-position, y-position, hue, marker style, and size). It also computes non-parametric bootstrap confidence intervals out of the box.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Map continuous and discrete variables to aesthetic dimensions using `sns.scatterplot()`.
2. Visualize time-series aggregations with automated bootstrap confidence intervals (`sns.lineplot()`).
3. Compare continuous features across categorical groups using `sns.boxplot()` and `sns.violinplot()`.
4. Construct categorical strip and swarm plots with jitter.
5. **Advanced:** Construct multi-facet relational grids (`sns.relplot`) and interaction point plots with bootstrap confidence intervals.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid")
print("Seaborn loaded!")

### 1. Multi-Dimensional Relational Mapping: `sns.scatterplot()`

Aesthetic dimensions in Seaborn:
- `x`, `y`: Primary coordinates.
- `hue`: Categorical or continuous color gradient.
- `style`: Discrete marker glyphs (e.g. circle, square, triangle).
- `size`: Marker area proportional to a numeric magnitude.

In [ ]:
# Synthetic vehicle efficiency dataset
n_cars = 200
horsepower = rng.uniform(60, 300, size=n_cars)
weight = horsepower * 12 + rng.normal(0, 300, size=n_cars) + 1200
mpg = 55 - (horsepower * 0.08 + weight * 0.005) + rng.normal(0, 2, size=n_cars)
origin = rng.choice(['USA', 'Europe', 'Japan'], size=n_cars, p=[0.5, 0.25, 0.25])
cylinders = rng.choice([4, 6, 8], size=n_cars, p=[0.4, 0.35, 0.25])

df_cars = pd.DataFrame({
    'Horsepower': horsepower,
    'Weight': weight,
    'MPG': mpg,
    'Origin': origin,
    'Cylinders': cylinders
})

fig, ax = plt.subplots(figsize=(9, 5.5))

sns.scatterplot(
    data=df_cars,
    x='Horsepower',
    y='MPG',
    hue='Origin',
    style='Cylinders',
    size='Weight',
    sizes=(30, 200),
    palette='Set1',
    alpha=0.85,
    ax=ax
)

ax.set_title("5D Vehicle Efficiency: HP vs. MPG by Origin, Cylinders, and Weight", fontsize=13, fontweight='bold')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

---
### 2. Time Series & Confidence Intervals: `sns.lineplot()`

When multiple experimental runs or trials share the same $x$ coordinate (e.g. multiple seeds of training loss over epochs):
- `sns.lineplot()` automatically computes and plots the **mean trajectory line**.
- It computes a **95% Bootstrap Confidence Interval** band (`errorbar='ci'` or `errorbar='sd'`).

In [ ]:
# Simulating 5 independent runs of 2 optimization algorithms
epochs = np.tile(np.arange(1, 26), 10)
run_ids = np.repeat(np.arange(10), 25)
algo = np.where(run_ids < 5, 'AdamW', 'SGD_Momentum')

# Simulate convergence with random noise per run
loss = np.zeros(len(epochs))
for i in range(len(epochs)):
    base = 2.0 * np.exp(-0.25 * epochs[i]) if algo[i] == 'AdamW' else 2.5 * np.exp(-0.15 * epochs[i])
    loss[i] = base + rng.normal(0, 0.08)

df_runs = pd.DataFrame({'Epoch': epochs, 'Loss': loss, 'Algorithm': algo, 'Run_ID': run_ids})

fig, ax = plt.subplots(figsize=(8.5, 4.5))

sns.lineplot(data=df_runs, x='Epoch', y='Loss', hue='Algorithm', errorbar='sd', linewidth=2.5, ax=ax)
ax.set_title("Optimizer Convergence Across 5 Random Seeds (Mean +/- Std Dev)", fontsize=13, fontweight='bold')

plt.show()

---
### 3. Categorical Distributions: Box Plots vs. Violin Plots

- **`sns.boxplot()`**: Displays the 5-number summary (Min, $Q_1$, Median, $Q_3$, Max) and outlier fliers.
- **`sns.violinplot()`**: Combines a box plot with a rotated KDE on each side, revealing hidden multi-modal structures that box plots conceal!

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Box Plot
sns.boxplot(data=df_cars, x='Origin', y='Horsepower', hue='Origin', palette='Set2', legend=False, ax=ax1)
ax1.set_title("Horsepower by Origin (Box Plot)", fontweight='bold')

# Violin Plot with inner quartiles
sns.violinplot(data=df_cars, x='Origin', y='Horsepower', hue='Origin', palette='Set2', legend=False, inner='quartile', ax=ax2)
ax2.set_title("Horsepower by Origin (Violin Density Plot)", fontweight='bold')

plt.tight_layout()
plt.show()

---
### 4. Advanced Complex Usage: Faceted Relational Analysis (`sns.relplot`) & Point Plots with Factor Interactions

In predictive modeling and diagnostic analysis:
- **`sns.relplot()`**: Combines scatter or line plots across multi-categorical grid facets.
- **`sns.pointplot()`**: Focuses strictly on comparing group means and confidence intervals, highlighting interactions between factors (slopes between levels).

In [ ]:
# 1. Faceted Relational Plot across Origin and Cylinders
g = sns.relplot(
    data=df_cars,
    x='Weight',
    y='MPG',
    hue='Origin',
    col='Cylinders',
    kind='scatter',
    palette='tab10',
    height=3.8,
    aspect=1.0
)
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle("Faceted Efficiency Analysis across Cylinder Counts", fontsize=13, fontweight='bold')
plt.show()

# 2. Factor Interaction with sns.pointplot
fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.pointplot(
    data=df_cars,
    x='Cylinders',
    y='MPG',
    hue='Origin',
    markers=['o', 's', '^'],
    linestyles=['-', '--', ':'],
    capsize=0.1,
    errorbar='ci',
    palette='Dark2',
    ax=ax
)
ax.set_title("Interaction Plot: Mean MPG by Cylinders and Origin (95% CI)", fontsize=13, fontweight='bold')
ax.set_ylabel("Mean MPG")
plt.show()

### Summary & Next Steps
In this notebook, you mastered:
- 5D multivariate relational mapping with `scatterplot`.
- Bootstrap confidence interval plotting with `lineplot`.
- Comparing continuous variables across categories using box, violin, and strip plots.
- Faceted multi-panel relational grids (`relplot`) and factor interaction estimation with `pointplot`.

**Next Notebook:** `03_matrix_plots_and_correlation_heatmaps.ipynb` — Correlation matrices, masked heatmaps, and hierarchical clustermaps for multicollinearity detection.